# Classificação de doença cardíaca com a API Keras

Este notebook implementa uma solução completa de deep learning para estimar a probabilidade de doença cardíaca a partir do dataset `heart.csv`.

A solução inclui pré-processamento de variáveis numéricas e categóricas com camadas Keras, treinamento, avaliação e inferência para um novo paciente.

In [ ]:
import os

# O backend deve ser definido antes de importar keras.
# TensorFlow é o padrão deste ambiente; torch também pode ser usado se instalado.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
# Limites conservadores de threads para execução estável em máquinas locais.
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("TF_DATA_PRIVATE_THREADPOOL_SIZE", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(1337)
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

## 1. Carregamento e inspeção dos dados

O alvo `target` vale 0 para ausência e 1 para presença de doença cardíaca.

In [ ]:
DATA_URL = "http://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
dataframe = pd.read_csv(DATA_URL)

assert "target" in dataframe.columns
print("Shape:", dataframe.shape)
display(dataframe.head())

In [ ]:
TARGET = "target"
NUMERIC_FEATURES = [
    "age", "trestbps", "chol", "thalach", "oldpeak", "slope"
]
CATEGORICAL_FEATURES = [
    "sex", "cp", "fbs", "restecg", "exang", "ca", "thal"
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Divisão determinística: 80% para treino e 20% para validação.
val_dataframe = dataframe.sample(frac=0.20, random_state=1337)
train_dataframe = dataframe.drop(val_dataframe.index)

y_train = train_dataframe[TARGET].to_numpy(dtype="float32").reshape(-1, 1)
y_val = val_dataframe[TARGET].to_numpy(dtype="float32").reshape(-1, 1)

print(f"Treino: {len(train_dataframe)} amostras")
print(f"Validação: {len(val_dataframe)} amostras")
print("Distribuição no treino:", np.bincount(y_train.astype("int32").ravel()))

## 2. Preparação das entradas

As variáveis numéricas serão normalizadas por camadas `Normalization`. As variáveis categóricas serão convertidas em índices e codificadas com `IntegerLookup(output_mode="one_hot")`. O vocabulário é aprendido somente no conjunto de treino.

In [ ]:
# Mapeamento treinado somente com os dados de treino. -1 representa categoria desconhecida.
categorical_vocabularies = {
    name: sorted(train_dataframe[name].astype(str).unique().tolist())
    for name in CATEGORICAL_FEATURES
}
categorical_to_index = {
    name: {value: index for index, value in enumerate(vocabulary)}
    for name, vocabulary in categorical_vocabularies.items()
}

def make_inputs(frame):
    inputs = {}
    for name in NUMERIC_FEATURES:
        inputs[name] = frame[name].to_numpy(dtype="float32").reshape(-1, 1)
    for name in CATEGORICAL_FEATURES:
        mapping = categorical_to_index[name]
        values = frame[name].astype(str).map(mapping).fillna(-1)
        inputs[name] = values.to_numpy(dtype="int32").reshape(-1, 1)
    return inputs

train_inputs = make_inputs(train_dataframe)
val_inputs = make_inputs(val_dataframe)
print({name: value.shape for name, value in train_inputs.items()})

## 3. Construção do modelo Keras

Será usado um modelo funcional com uma entrada nomeada para cada característica. A saída sigmoid representa a probabilidade estimada de doença cardíaca.

In [ ]:
def build_model(train_frame):
    inputs = {}
    encoded_features = []

    for name in NUMERIC_FEATURES:
        feature_input = keras.Input(shape=(1,), name=name, dtype="float32")
        normalizer = layers.Normalization(axis=-1, name=f"{name}_normalization")
        normalizer.adapt(
            train_frame[name].to_numpy(dtype="float32").reshape(-1, 1)
        )
        inputs[name] = feature_input
        encoded_features.append(normalizer(feature_input))

    for name in CATEGORICAL_FEATURES:
        feature_input = keras.Input(shape=(1,), name=name, dtype="int32")
        lookup = layers.IntegerLookup(
            vocabulary=list(range(len(categorical_vocabularies[name]))),
            num_oov_indices=1,
            output_mode="one_hot",
            name=f"{name}_lookup",
        )
        inputs[name] = feature_input
        encoded_features.append(lookup(feature_input))

    x = layers.Concatenate(name="all_features")(encoded_features)
    x = layers.Dense(64, activation="relu", name="dense_1")(x)
    x = layers.Dropout(0.30, name="dropout")(x)
    x = layers.Dense(32, activation="relu", name="dense_2")(x)
    output = layers.Dense(1, activation="sigmoid", name="probability")(x)

    return keras.Model(inputs=inputs, outputs=output, name="heart_disease_classifier")

model = build_model(train_dataframe)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)
model.summary()

## 4. Treinamento

O `EarlyStopping` interrompe o treinamento quando a loss de validação deixa de melhorar e restaura os melhores pesos.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=12, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5
    ),
]

history = model.fit(
    train_inputs,
    y_train,
    validation_data=(val_inputs, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

## 5. Avaliação e decisão

A rede produz uma probabilidade. Para transformar essa probabilidade em classe, usamos o limiar `0.5`.

In [ ]:
results = model.evaluate(val_inputs, y_val, return_dict=True, verbose=0)
print({name: round(float(value), 4) for name, value in results.items()})

probabilities = model.predict(val_inputs, verbose=0).ravel()
threshold = 0.5
predicted_classes = (probabilities >= threshold).astype("int32")
true_classes = y_val.ravel().astype("int32")

tp = int(np.sum((predicted_classes == 1) & (true_classes == 1)))
tn = int(np.sum((predicted_classes == 0) & (true_classes == 0)))
fp = int(np.sum((predicted_classes == 1) & (true_classes == 0)))
fn = int(np.sum((predicted_classes == 0) & (true_classes == 1)))
print(f"Matriz de confusão: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

In [ ]:
comparison = pd.DataFrame(
    {
        "real": true_classes,
        "probabilidade": probabilities,
        "classe": predicted_classes,
    }
)
comparison.head(10)

## 6. Inferência para um novo paciente

A função abaixo aplica exatamente o mesmo mapeamento categórico usado no treino e retorna a probabilidade prevista.

In [ ]:
def make_single_input(sample):
    sample_frame = pd.DataFrame([sample])
    return make_inputs(sample_frame)

sample = {
    "age": 60,
    "sex": 1,
    "cp": 1,
    "trestbps": 145,
    "chol": 233,
    "fbs": 1,
    "restecg": 2,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 2.3,
    "slope": 3,
    "ca": 0,
    "thal": "fixed",
}

sample_probability = float(model.predict(make_single_input(sample), verbose=0)[0, 0])
sample_class = int(sample_probability >= threshold)
print(f"Probabilidade estimada: {sample_probability:.3f}")
print(f"Classe prevista: {sample_class}")

## Conclusão

A solução usa um modelo Keras funcional para receber as 13 características, normalizar valores numéricos, codificar categorias, aprender uma representação não linear e gerar uma probabilidade binária. A qualidade final deve ser analisada pela métrica de validação, especialmente AUC, além da acurácia.